# NBA offensive archetypes — reproducible clustering

This notebook groups NBA players by **how they create offense**, not simply by how many points they score. It fixes the original analysis by:

- pinning one season and caching a single data snapshot;
- joining every source with `PLAYER_ID`;
- enforcing games, minutes, and tracked-possession thresholds before modeling;
- using play-type possessions and mutually exclusive shot-zone attempts instead of play-type points;
- distinguishing missing data from genuine zero usage;
- evaluating both separation and bootstrap stability before choosing the cluster count;
- clustering in the full feature space and using PCA only for visualization; and
- describing clusters with actual player means and representative players.

Dependencies: `nba_api`, `pandas`, `numpy`, `scikit-learn`, `matplotlib`, and `seaborn`. If needed, run `%pip install nba_api pandas numpy scikit-learn matplotlib seaborn` once in your kernel.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nba_api.stats import endpoints
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

# Reproducible data and modeling configuration
SEASON = "2025-26"
SEASON_TYPE = "Regular Season"
MIN_GP = 20
MIN_MPG = 15
MIN_TRACKED_POSS = 100
RANDOM_STATE = 42
K_VALUES = range(2, 13)
SELECTED_K = 6  # Style-first scoring taxonomy; height and efficiency are excluded.
N_BOOTSTRAPS = 20
MIN_SUBTYPE_SIZE = 12
SUBTYPE_MIN_SILHOUETTE = 0.10
SUBTYPE_MIN_STABILITY = 0.65

CACHE_DIR = Path("archetypes_data") / SEASON.replace("-", "_")
OUTPUT_DIR = Path("archetypes_outputs") / SEASON.replace("-", "_")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REFRESH_DATA = False

# NBA Synergy's API spelling and our stable internal labels.
PLAY_TYPES = {
    "Transition": "Transition",
    "Isolation": "Isolation",
    "PRBallHandler": "PRBallHandler",
    "PRRollman": "PRRollMan",
    "OffRebound": "OffRebound",
    "Spotup": "Spotup",
    "Cut": "Cut",
    "Handoff": "Handoff",
    "OffScreen": "OffScreen",
    "Misc": "Misc",
    "Postup": "Postup",
}
PLAY_KEYS = list(PLAY_TYPES.values())

print(f"Season: {SEASON} | Eligibility: {MIN_GP}+ GP, {MIN_MPG}+ MPG, {MIN_TRACKED_POSS}+ tracked possessions")

## 1. Fetch and cache one coherent snapshot

Every endpoint receives the same explicit season and season type. Cached CSVs make reruns deterministic; set `REFRESH_DATA = True` only when you intentionally want a new snapshot.

In [ ]:
def fetch_cached(name, fetch_fn, refresh=REFRESH_DATA, attempts=3):
    path = CACHE_DIR / f"{name}.csv"
    if path.exists() and not refresh:
        return pd.read_csv(path)

    last_error = None
    for attempt in range(attempts):
        try:
            frame = fetch_fn()
            frame.to_csv(path, index=False)
            return frame
        except Exception as error:
            last_error = error
            if attempt < attempts - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Could not fetch {name} after {attempts} attempts") from last_error


def first_frame(endpoint):
    return endpoint.get_data_frames()[0]


def fetch_shot_zones():
    frame = first_frame(
        endpoints.LeagueDashPlayerShotLocations(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            distance_range="By Zone",
            per_mode_detailed="Totals",
            timeout=60,
        )
    )
    frame.columns = [
        str(second) if not str(first) else f"{first}_{second}"
        for first, second in frame.columns
    ]
    return frame


base_df = fetch_cached(
    "player_base",
    lambda: first_frame(
        endpoints.LeagueDashPlayerStats(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            measure_type_detailed_defense="Base",
            per_mode_detailed="Totals",
            timeout=60,
        )
    ),
)

usage_df = fetch_cached(
    "player_usage",
    lambda: first_frame(
        endpoints.LeagueDashPlayerStats(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            measure_type_detailed_defense="Usage",
            per_mode_detailed="Totals",
            timeout=60,
        )
    ),
)

scoring_style_df = fetch_cached(
    "player_scoring_style",
    lambda: first_frame(
        endpoints.LeagueDashPlayerStats(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            measure_type_detailed_defense="Scoring",
            per_mode_detailed="Totals",
            timeout=60,
        )
    ),
)

catch_shoot_df = fetch_cached(
    "player_catch_shoot",
    lambda: first_frame(
        endpoints.LeagueDashPlayerPtShot(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            general_range_nullable="Catch and Shoot",
            per_mode_simple="Totals",
            timeout=60,
        )
    ),
)

pullup_df = fetch_cached(
    "player_pullups",
    lambda: first_frame(
        endpoints.LeagueDashPlayerPtShot(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            general_range_nullable="Pullups",
            per_mode_simple="Totals",
            timeout=60,
        )
    ),
)

drives_df = fetch_cached(
    "player_drives",
    lambda: first_frame(
        endpoints.LeagueDashPtStats(
            season=SEASON,
            season_type_all_star=SEASON_TYPE,
            player_or_team="Player",
            pt_measure_type="Drives",
            per_mode_simple="Totals",
            timeout=60,
        )
    ),
)

shot_zone_df = fetch_cached("shot_zones", fetch_shot_zones)

synergy_frames = []
for api_name, play_key in PLAY_TYPES.items():
    frame = fetch_cached(
        f"synergy_{play_key}",
        lambda api_name=api_name: first_frame(
            endpoints.SynergyPlayTypes(
                season=SEASON,
                season_type_all_star=SEASON_TYPE,
                per_mode_simple="Totals",
                player_or_team_abbreviation="P",
                type_grouping_nullable="Offensive",
                play_type_nullable=api_name,
                timeout=60,
            )
        ),
    ).copy()
    frame["PLAY_TYPE_KEY"] = play_key
    synergy_frames.append(frame)

synergy_df = pd.concat(synergy_frames, ignore_index=True)

print(
    f"Loaded {len(base_df):,} player rows, {len(shot_zone_df):,} shot-zone rows, "
    f"and {len(synergy_df):,} play-type rows."
)

## 2. Define the eligible population, then join by player ID

The original left join allowed low-minute players to remain and then assigned them average values. Here, eligibility is applied first. Required fields are dropped when unavailable; only an absent play-type row is treated as zero, because it represents no recorded possessions for that play type within an otherwise sufficiently large tracked sample.

In [ ]:
eligible = base_df[
    ["PLAYER_ID", "PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "MIN"]
].copy()
eligible["MPG"] = eligible["MIN"] / eligible["GP"].replace(0, np.nan)
eligible = eligible.query("GP >= @MIN_GP and MPG >= @MIN_MPG").copy()
eligible = eligible.drop_duplicates("PLAYER_ID")

play_counts = (
    synergy_df.pivot_table(
        index="PLAYER_ID",
        columns="PLAY_TYPE_KEY",
        values="POSS",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=PLAY_KEYS, fill_value=0)
    .rename(columns=lambda value: f"PLAY_POSS_{value}")
    .reset_index()
)
play_count_cols = [f"PLAY_POSS_{value}" for value in PLAY_KEYS]

usage = usage_df[["PLAYER_ID", "USG_PCT"]].drop_duplicates("PLAYER_ID")
scoring_style = scoring_style_df[
    ["PLAYER_ID", "PCT_UAST_FGM", "PCT_PTS_FT"]
].drop_duplicates("PLAYER_ID")
catch_shoot = catch_shoot_df[["PLAYER_ID", "FGA_FREQUENCY"]].rename(
    columns={"FGA_FREQUENCY": "CATCH_SHOOT_FGA_FREQ"}
).drop_duplicates("PLAYER_ID")
pullups = pullup_df[["PLAYER_ID", "FGA_FREQUENCY"]].rename(
    columns={"FGA_FREQUENCY": "PULLUP_FGA_FREQ"}
).drop_duplicates("PLAYER_ID")
drives = drives_df[["PLAYER_ID", "DRIVES"]].drop_duplicates("PLAYER_ID")
shot_counts = pd.DataFrame(
    {
        "PLAYER_ID": shot_zone_df["PLAYER_ID"],
        "SHOT_FGA_RIM": shot_zone_df["Restricted Area_FGA"],
        "SHOT_FGA_PAINT": shot_zone_df["In The Paint (Non-RA)_FGA"],
        "SHOT_FGA_MIDRANGE": shot_zone_df["Mid-Range_FGA"],
        "SHOT_FGA_CORNER3": (
            shot_zone_df["Left Corner 3_FGA"] + shot_zone_df["Right Corner 3_FGA"]
        ),
        "SHOT_FGA_ABOVE_BREAK3": shot_zone_df["Above the Break 3_FGA"],
    }
).groupby("PLAYER_ID", as_index=False).sum()
shot_count_cols = [column for column in shot_counts if column.startswith("SHOT_FGA_")]

model_df = (
    eligible.merge(play_counts, on="PLAYER_ID", how="inner", validate="one_to_one")
    .merge(usage, on="PLAYER_ID", how="left", validate="one_to_one")
    .merge(scoring_style, on="PLAYER_ID", how="left", validate="one_to_one")
    .merge(catch_shoot, on="PLAYER_ID", how="left", validate="one_to_one")
    .merge(pullups, on="PLAYER_ID", how="left", validate="one_to_one")
    .merge(drives, on="PLAYER_ID", how="left", validate="one_to_one")
    .merge(shot_counts, on="PLAYER_ID", how="left", validate="one_to_one")
)

# Missing catch-and-shoot or pull-up rows mean no recorded attempts in that style.
model_df[["CATCH_SHOOT_FGA_FREQ", "PULLUP_FGA_FREQ"]] = model_df[
    ["CATCH_SHOOT_FGA_FREQ", "PULLUP_FGA_FREQ"]
].fillna(0)
model_df["TRACKED_POSS"] = model_df[play_count_cols].sum(axis=1)
model_df["TOTAL_ZONE_FGA"] = model_df[shot_count_cols].sum(axis=1)
model_df["DRIVES_PER_36"] = model_df["DRIVES"] / model_df["MIN"] * 36

required_columns = [
    "USG_PCT",
    "PCT_UAST_FGM",
    "PCT_PTS_FT",
    "DRIVES_PER_36",
    *shot_count_cols,
]
missing_before_drop = model_df[required_columns].isna().sum().rename("missing_rows")

model_df = model_df.dropna(subset=required_columns).copy()
model_df = model_df.query(
    "TRACKED_POSS >= @MIN_TRACKED_POSS and TOTAL_ZONE_FGA > 0"
).reset_index(drop=True)

display(missing_before_drop.to_frame())
display(
    pd.Series(
        {
            "eligible_before_feature_checks": len(eligible),
            "final_model_players": len(model_df),
            "median_games": model_df["GP"].median(),
            "median_mpg": model_df["MPG"].median(),
            "median_tracked_possessions": model_df["TRACKED_POSS"].median(),
        },
        name="sample",
    ).to_frame()
)

## 3. Build role features

Play types and shot zones are compositions: their shares add to 100%. A small count-based pseudocount allows a centered log-ratio (CLR) transform without turning missing data into fake league averages. Each feature family is standardized and weighted so the eleven play-type fields do not overwhelm the five shot-zone fields simply because there are more of them.

In [ ]:
def composition_features(frame, count_columns, prefix, pseudocount=0.5):
    counts = frame[count_columns].to_numpy(dtype=float)
    totals = counts.sum(axis=1, keepdims=True)
    shares = np.divide(counts, totals, out=np.zeros_like(counts), where=totals > 0)

    smoothed = counts + pseudocount
    smoothed /= smoothed.sum(axis=1, keepdims=True)
    clr = np.log(smoothed) - np.log(smoothed).mean(axis=1, keepdims=True)

    labels = [column.split("_", 2)[-1] for column in count_columns]
    share_columns = [f"{prefix}_SHARE_{label}" for label in labels]
    clr_columns = [f"{prefix}_CLR_{label}" for label in labels]
    frame[share_columns] = shares
    frame[clr_columns] = clr
    return share_columns, clr_columns


play_share_cols, play_clr_cols = composition_features(
    model_df, play_count_cols, "PLAY"
)
shot_share_cols, shot_clr_cols = composition_features(
    model_df, shot_count_cols, "SHOT"
)

model_df["TRACKED_POSS_PER_36"] = model_df["TRACKED_POSS"] / model_df["MIN"] * 36
creation_style_cols = [
    "CATCH_SHOOT_FGA_FREQ",
    "PULLUP_FGA_FREQ",
    "PCT_UAST_FGM",
    "PCT_PTS_FT",
    "DRIVES_PER_36",
]
volume_cols = ["USG_PCT", "TRACKED_POSS_PER_36"]


def standardized_family(columns):
    values = StandardScaler().fit_transform(model_df[columns])
    return values / np.sqrt(len(columns))


X = np.hstack(
    [
        standardized_family(play_clr_cols),
        standardized_family(shot_clr_cols),
        standardized_family(creation_style_cols),
        standardized_family(volume_cols),
    ]
)

assert np.isfinite(X).all()
print(f"Model matrix: {X.shape[0]} players × {X.shape[1]} features")

## 4. Evaluate the number of archetypes

Silhouette measures separation, Calinski–Harabasz and Davies–Bouldin provide complementary compactness checks, and bootstrap adjusted Rand index (ARI) tests whether clusters survive modest changes to the player sample. `SELECTED_K` remains an explicit modeling decision rather than being silently overridden by one metric.

In [ ]:
def bootstrap_stability(X, reference_labels, k, n_bootstraps=N_BOOTSTRAPS):
    rng = np.random.default_rng(RANDOM_STATE + k)
    sample_size = int(0.80 * len(X))
    scores = []

    for bootstrap in range(n_bootstraps):
        sample = rng.choice(len(X), size=sample_size, replace=False)
        candidate = KMeans(
            n_clusters=k,
            n_init=50,
            random_state=RANDOM_STATE + 1000 * k + bootstrap,
        ).fit(X[sample])
        scores.append(adjusted_rand_score(reference_labels, candidate.predict(X)))

    return float(np.mean(scores)), float(np.std(scores))


diagnostic_rows = []
for k in K_VALUES:
    candidate = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(X)
    labels = candidate.labels_
    stability_mean, stability_std = bootstrap_stability(X, labels, k)
    diagnostic_rows.append(
        {
            "k": k,
            "silhouette": silhouette_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels),
            "davies_bouldin": davies_bouldin_score(X, labels),
            "bootstrap_ari_mean": stability_mean,
            "bootstrap_ari_std": stability_std,
            "smallest_cluster": int(pd.Series(labels).value_counts().min()),
        }
    )

diagnostics_df = pd.DataFrame(diagnostic_rows).set_index("k")
display(diagnostics_df.round(3))

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
metric_specs = [
    ("silhouette", "Silhouette (higher is better)"),
    ("bootstrap_ari_mean", "Bootstrap stability / ARI (higher is better)"),
    ("calinski_harabasz", "Calinski–Harabasz (higher is better)"),
    ("davies_bouldin", "Davies–Bouldin (lower is better)"),
]
for axis, (metric, title) in zip(axes.flat, metric_specs):
    axis.plot(diagnostics_df.index, diagnostics_df[metric], marker="o")
    axis.axvline(SELECTED_K, color="tab:red", linestyle="--", label=f"Selected k={SELECTED_K}")
    axis.set_title(title)
    axis.set_xlabel("Number of clusters")
    axis.legend()
plt.tight_layout()
plt.show()

## 5. Fit the selected model and inspect overlap

K-means is fit on the full weighted feature space. PCA below is only a two-dimensional view; overlap in this chart does not change the fitted assignments. The confidence margin compares each player's nearest and second-nearest centroid—small values identify hybrid or uncertain assignments.

In [ ]:
kmeans = KMeans(
    n_clusters=SELECTED_K,
    n_init=100,
    random_state=RANDOM_STATE,
).fit(X)
model_df["CLUSTER"] = kmeans.labels_

distances = kmeans.transform(X)
nearest_two = np.sort(distances, axis=1)[:, :2]
model_df["DISTANCE_TO_CENTROID"] = nearest_two[:, 0]
model_df["CONFIDENCE_MARGIN"] = (
    nearest_two[:, 1] - nearest_two[:, 0]
) / nearest_two[:, 1]

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
coordinates = pca_2d.fit_transform(X)

plt.figure(figsize=(12, 8))
cluster_cmap = plt.get_cmap("tab10", SELECTED_K)
scatter = plt.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=model_df["CLUSTER"],
    cmap=cluster_cmap,
    vmin=-0.5,
    vmax=SELECTED_K - 0.5,
    alpha=0.80,
)
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
plt.title(f"NBA offensive archetypes, {SEASON} — PCA view only")
plt.colorbar(scatter, label="Cluster", ticks=range(SELECTED_K))
plt.show()

display(model_df.groupby("CLUSTER").size().rename("players").to_frame())

## 6. Interpret clusters from real player averages

These profiles are direct means of the assigned players—not inverse-PCA reconstructions—so percentages cannot become negative. Archetype names are generated from the two most overrepresented traits and therefore stay synchronized if the model is rerun.

In [ ]:
profile_cols = [
    *play_share_cols,
    *shot_share_cols,
    *creation_style_cols,
    *volume_cols,
]
profiles_df = model_df.groupby("CLUSTER")[profile_cols].mean()
profile_z = (profiles_df - model_df[profile_cols].mean()) / model_df[profile_cols].std(ddof=0)

pretty_feature = {
    "PLAY_SHARE_Transition": "Transition",
    "PLAY_SHARE_Isolation": "Isolation",
    "PLAY_SHARE_PRBallHandler": "P&R ball handler",
    "PLAY_SHARE_PRRollMan": "P&R roll man",
    "PLAY_SHARE_OffRebound": "Offensive rebounding",
    "PLAY_SHARE_Spotup": "Spot-up",
    "PLAY_SHARE_Cut": "Cutting",
    "PLAY_SHARE_Handoff": "Handoff",
    "PLAY_SHARE_OffScreen": "Off-screen",
    "PLAY_SHARE_Misc": "Miscellaneous",
    "PLAY_SHARE_Postup": "Post-up",
    "SHOT_SHARE_RIM": "Rim attempts",
    "SHOT_SHARE_PAINT": "Paint attempts",
    "SHOT_SHARE_MIDRANGE": "Mid-range attempts",
    "SHOT_SHARE_CORNER3": "Corner-three attempts",
    "SHOT_SHARE_ABOVE_BREAK3": "Above-break-three attempts",
    "CATCH_SHOOT_FGA_FREQ": "Catch-and-shoot frequency",
    "PULLUP_FGA_FREQ": "Pull-up frequency",
    "PCT_UAST_FGM": "Unassisted scoring",
    "PCT_PTS_FT": "Free-throw scoring share",
    "DRIVES_PER_36": "Drives per 36",
    "USG_PCT": "High usage",
    "TRACKED_POSS_PER_36": "High involvement",
}

archetype_names = {}
if SELECTED_K == 6:
    # Assign stable basketball role names from the measured cluster profiles,
    # rather than tying names to arbitrary numeric cluster IDs.
    unassigned = set(profiles_df.index)

    def take_highest(score):
        cluster = score.loc[sorted(unassigned)].idxmax()
        unassigned.remove(cluster)
        return cluster

    on_ball = take_highest(profiles_df["PLAY_SHARE_PRBallHandler"])
    rim_runner = take_highest(
        profiles_df["SHOT_SHARE_RIM"]
        + profiles_df["PLAY_SHARE_OffRebound"]
        + profiles_df["PLAY_SHARE_Cut"]
    )
    self_creator = take_highest(profiles_df["USG_PCT"])
    perimeter_shooter = take_highest(profiles_df["SHOT_SHARE_ABOVE_BREAK3"])
    role_forward = take_highest(profiles_df["SHOT_SHARE_CORNER3"])
    combo_forward = unassigned.pop()

    archetype_names = {
        on_ball: "On-ball pick-and-roll creators",
        rim_runner: "Rim-running centers",
        self_creator: "High-usage self-creators",
        perimeter_shooter: "Perimeter shooters",
        role_forward: "Low-usage role forwards",
        combo_forward: "Versatile combo forwards",
    }
else:
    for cluster, row in profile_z.iterrows():
        top_play_trait = row[play_share_cols].idxmax()
        top_supporting_trait = row[[*shot_share_cols, *creation_style_cols, *volume_cols]].idxmax()
        archetype_names[cluster] = (
            f"{pretty_feature[top_play_trait]} + {pretty_feature[top_supporting_trait]}"
        )

model_df["ARCHETYPE"] = model_df["CLUSTER"].map(archetype_names)
profiles_df.insert(0, "ARCHETYPE", profiles_df.index.map(archetype_names))

representative_rows = []
for cluster in range(SELECTED_K):
    examples = (
        model_df.loc[model_df["CLUSTER"] == cluster]
        .nsmallest(8, "DISTANCE_TO_CENTROID")["PLAYER_NAME"]
        .tolist()
    )
    representative_rows.append(
        {
            "CLUSTER": cluster,
            "ARCHETYPE": archetype_names[cluster],
            "PLAYERS": int((model_df["CLUSTER"] == cluster).sum()),
            "REPRESENTATIVE_PLAYERS": ", ".join(examples),
        }
    )

cluster_summary_df = pd.DataFrame(representative_rows).set_index("CLUSTER")
display(cluster_summary_df)

plt.figure(figsize=(16, max(5, SELECTED_K * 0.8)))
heatmap_df = profile_z.rename(columns=pretty_feature).copy()
heatmap_df.index = [f"{cluster}: {archetype_names[cluster]}" for cluster in heatmap_df.index]
sns.heatmap(
    heatmap_df,
    cmap="vlag",
    center=0,
    linewidths=0.4,
    cbar_kws={"label": "Standard deviations from league sample"},
)
plt.title("Archetype profiles — red is above the sample average")
plt.xlabel("Trait")
plt.ylabel("Archetype")
plt.tight_layout()
plt.show()

# Add a betting-oriented second level without forcing one fragile flat k.
# A broad family is split only when both children are large and reproducible.
model_df["SUBTYPE_ID"] = -1
model_df["SUBTYPE_ARCHETYPE"] = ""
model_df["SUBTYPE_DISTANCE_TO_CENTROID"] = np.nan
model_df["SUBTYPE_CONFIDENCE_MARGIN"] = np.nan
subtype_rows = []
next_subtype_id = 0
detail_columns = [*creation_style_cols, *play_share_cols, *shot_share_cols]

for parent_cluster in sorted(model_df["CLUSTER"].unique()):
    parent_positions = np.flatnonzero(
        model_df["CLUSTER"].to_numpy() == parent_cluster
    )
    parent_X = X[parent_positions]
    parent_profile = model_df.loc[parent_positions, profile_cols]
    split_model = None
    split_silhouette = np.nan
    split_stability = np.nan

    if len(parent_positions) >= 2 * MIN_SUBTYPE_SIZE:
        candidate = KMeans(
            n_clusters=2,
            n_init=100,
            random_state=RANDOM_STATE + int(parent_cluster),
        ).fit(parent_X)
        candidate_sizes = pd.Series(candidate.labels_).value_counts()
        split_silhouette = silhouette_score(parent_X, candidate.labels_)
        split_stability, _ = bootstrap_stability(parent_X, candidate.labels_, 2)
        if (
            candidate_sizes.min() >= MIN_SUBTYPE_SIZE
            and split_silhouette >= SUBTYPE_MIN_SILHOUETTE
            and split_stability >= SUBTYPE_MIN_STABILITY
        ):
            split_model = candidate

    if split_model is None:
        local_labels = np.zeros(len(parent_positions), dtype=int)
        local_distances = model_df.loc[
            parent_positions, "DISTANCE_TO_CENTROID"
        ].to_numpy()
        local_confidence = np.full(len(parent_positions), np.nan)
    else:
        local_labels = split_model.labels_
        subtype_distances = split_model.transform(parent_X)
        nearest_two = np.sort(subtype_distances, axis=1)[:, :2]
        local_distances = nearest_two[:, 0]
        local_confidence = (
            (nearest_two[:, 1] - nearest_two[:, 0]) / nearest_two[:, 1]
        )

    for child_label in sorted(np.unique(local_labels)):
        child_mask = local_labels == child_label
        child_positions = parent_positions[child_mask]

        if split_model is None:
            subtype_name = archetype_names[parent_cluster]
        else:
            child_profile = model_df.loc[child_positions, profile_cols].mean()
            parent_mean = parent_profile.mean()
            parent_name = archetype_names[parent_cluster]
            if parent_name == "On-ball pick-and-roll creators":
                subtype_name = (
                    "Rim-pressure P&R creators"
                    if child_profile["DRIVES_PER_36"] > parent_mean["DRIVES_PER_36"]
                    else "P&R shooting creators"
                )
            elif parent_name == "Low-usage role forwards":
                interior_score = (
                    child_profile["PLAY_SHARE_PRRollMan"]
                    + child_profile["PLAY_SHARE_Postup"]
                )
                parent_interior = (
                    parent_mean["PLAY_SHARE_PRRollMan"]
                    + parent_mean["PLAY_SHARE_Postup"]
                )
                subtype_name = (
                    "Interior role finishers"
                    if interior_score > parent_interior
                    else "Corner spot-up forwards"
                )
            elif parent_name == "Perimeter shooters":
                movement_score = (
                    child_profile["CATCH_SHOOT_FGA_FREQ"]
                    + child_profile["PLAY_SHARE_OffScreen"]
                )
                parent_movement = (
                    parent_mean["CATCH_SHOOT_FGA_FREQ"]
                    + parent_mean["PLAY_SHARE_OffScreen"]
                )
                subtype_name = (
                    "Movement and catch-and-shoot specialists"
                    if movement_score > parent_movement
                    else "Secondary-drive perimeter scorers"
                )
            elif parent_name == "High-usage self-creators":
                subtype_name = (
                    "Pull-up primary creators"
                    if child_profile["PULLUP_FGA_FREQ"] > parent_mean["PULLUP_FGA_FREQ"]
                    else "Power scoring forwards"
                )
            else:
                parent_std = parent_profile.std(ddof=0).replace(0, np.nan)
                child_contrast = (
                    (child_profile - parent_mean) / parent_std
                ).fillna(0)
                top_details = child_contrast[detail_columns].nlargest(2).index
                subtype_name = (
                    f"{parent_name} — "
                    + " / ".join(pretty_feature[column] for column in top_details)
                )

        model_df.loc[child_positions, "SUBTYPE_ID"] = next_subtype_id
        model_df.loc[child_positions, "SUBTYPE_ARCHETYPE"] = subtype_name
        model_df.loc[child_positions, "SUBTYPE_DISTANCE_TO_CENTROID"] = (
            local_distances[child_mask]
        )
        model_df.loc[child_positions, "SUBTYPE_CONFIDENCE_MARGIN"] = (
            local_confidence[child_mask]
        )

        representatives = (
            model_df.loc[child_positions]
            .nsmallest(6, "SUBTYPE_DISTANCE_TO_CENTROID")["PLAYER_NAME"]
            .tolist()
        )
        subtype_rows.append(
            {
                "SUBTYPE_ID": next_subtype_id,
                "PARENT_CLUSTER": parent_cluster,
                "PARENT_ARCHETYPE": archetype_names[parent_cluster],
                "SUBTYPE_ARCHETYPE": subtype_name,
                "PLAYERS": len(child_positions),
                "PARENT_SPLIT_SILHOUETTE": split_silhouette,
                "PARENT_SPLIT_STABILITY": split_stability,
                "REPRESENTATIVE_PLAYERS": ", ".join(representatives),
            }
        )
        next_subtype_id += 1

subtype_summary_df = pd.DataFrame(subtype_rows).set_index("SUBTYPE_ID")
model_df["SUBTYPE_ID"] = model_df["SUBTYPE_ID"].astype(int)
subtype_profiles_df = (
    model_df.groupby(["SUBTYPE_ID", "SUBTYPE_ARCHETYPE"])[profile_cols]
    .mean()
)
display(subtype_summary_df.round(3))

## 7. Save auditable outputs

The player assignment file now includes both broad families and finer scoring subtypes. The feature file exposes every model input for matchup and prop research. Low confidence margins should be treated as hybrid players rather than definitive classifications.

In [ ]:
player_output_cols = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "TEAM_ABBREVIATION",
    "GP",
    "MPG",
    "TRACKED_POSS",
    "CLUSTER",
    "ARCHETYPE",
    "DISTANCE_TO_CENTROID",
    "CONFIDENCE_MARGIN",
    "SUBTYPE_ID",
    "SUBTYPE_ARCHETYPE",
    "SUBTYPE_DISTANCE_TO_CENTROID",
    "SUBTYPE_CONFIDENCE_MARGIN",
]

player_output_path = OUTPUT_DIR / "player_archetypes.csv"
feature_output_path = OUTPUT_DIR / "player_scoring_features.csv"
profile_output_path = OUTPUT_DIR / "cluster_profiles.csv"
subtype_summary_path = OUTPUT_DIR / "subtype_summary.csv"
subtype_profile_path = OUTPUT_DIR / "subtype_profiles.csv"
diagnostic_output_path = OUTPUT_DIR / "cluster_diagnostics.csv"

model_df[player_output_cols].sort_values(
    ["CLUSTER", "SUBTYPE_ID", "SUBTYPE_DISTANCE_TO_CENTROID"]
).to_csv(player_output_path, index=False)
model_df[[*player_output_cols, *profile_cols]].sort_values(
    ["CLUSTER", "SUBTYPE_ID", "SUBTYPE_DISTANCE_TO_CENTROID"]
).to_csv(feature_output_path, index=False)
profiles_df.to_csv(profile_output_path)
subtype_summary_df.to_csv(subtype_summary_path)
subtype_profiles_df.to_csv(subtype_profile_path)
diagnostics_df.to_csv(diagnostic_output_path)

print(f"Saved {player_output_path}")
print(f"Saved {feature_output_path}")
print(f"Saved {profile_output_path}")
print(f"Saved {subtype_summary_path}")
print(f"Saved {subtype_profile_path}")
print(f"Saved {diagnostic_output_path}")
import json
import sys

sys.path.insert(0, str(Path("scripts").resolve()))
from matchup_analysis import compute_input_data_identity

# Bind this clustering execution to the exact membership and feature snapshots
# it just wrote, so the matchup run's stale-model guard compares against a
# digest recorded here rather than recomputing one from whatever files
# happen to coexist.
clustering_metadata = {
    "season": SEASON,
    "feature_definition": (
        "Eleven play-type and five shot-zone composition shares with centered "
        "log-ratio (0.5 pseudocount), catch-and-shoot and pull-up frequency, "
        "unassisted-scoring share, free-throw scoring share, drives per 36, "
        "usage, and tracked possessions per 36; each family standardized and "
        "weighted equally by family size"
    ),
    "clustering_method": (
        "KMeans(n_init=100) on the full weighted feature space, then conditional "
        "per-parent two-cluster KMeans(n_init=100) splits gated on minimum "
        "subtype size, silhouette score, and bootstrap stability; split and "
        "bootstrap seeds derive deterministically from the base random seed"
    ),
    "random_seed": int(RANDOM_STATE),
    "top_level_clusters": int(SELECTED_K),
    "cluster_count": int(model_df["SUBTYPE_ID"].nunique()),
    "input_data_identity": compute_input_data_identity(
        pd.read_csv(player_output_path), pd.read_csv(feature_output_path)
    ),
}
metadata_path = OUTPUT_DIR / "clustering_metadata.json"
metadata_path.write_text(json.dumps(clustering_metadata, indent=2, sort_keys=True))
print(f"Saved {metadata_path}")
